> **Chapter 15, Part 3** | Engineering lens. **Focus:** trigger work when a condition becomes true (a sensor), and flag an asset as stale when it falls behind a freshness target.

# Sensors and Freshness

A schedule says "run at 2am". A **sensor** says "run when the file lands". Sensors are how a pipeline reacts to the world instead of guessing when the world will be ready. A **freshness policy** is the other half: it tells you when an asset has fallen too far behind to trust.

Both need a notion of time. To keep the notebook deterministic and offline, we drive a simulated clock instead of `time.time()`. Nothing here sleeps; we just advance a counter and watch the sensor and the freshness check react.

In [1]:
from dataclasses import dataclass, field
from typing import Callable


class Clock:
    def __init__(self, t=0):
        self.t = t

    def tick(self, n=1):
        self.t += n
        return self.t


@dataclass
class Sensor:
    name: str
    condition: Callable   # condition(clock) -> bool
    on_fire: Callable     # on_fire(clock) -> None
    last_fired: object = None

    def poll(self, clock):
        if self.condition(clock):
            self.on_fire(clock)
            self.last_fired = clock.t
            return True
        return False


print("Sensor and Clock defined")

Sensor and Clock defined


A worked sensor: a file lands at tick 3 and again at tick 7. The sensor polls every tick and fires only when an unprocessed file is present. This is the polling loop every orchestrator runs under the hood.

In [2]:
clock = Clock()
arrivals = {3, 7}          # ticks at which a new file appears
processed = set()
runs = []


def file_waiting(clk):
    return clk.t in arrivals and clk.t not in processed


def process_file(clk):
    processed.add(clk.t)
    runs.append(clk.t)


sensor = Sensor("new_file", file_waiting, process_file)

for _ in range(10):
    t = clock.t
    fired = sensor.poll(clock)
    print(f"  t={t}: " + ("FIRED -> materialized partition" if fired else "idle"))
    clock.tick()

print()
print("sensor fired at ticks:", runs)

  t=0: idle
  t=1: idle
  t=2: idle
  t=3: FIRED -> materialized partition
  t=4: idle
  t=5: idle
  t=6: idle
  t=7: FIRED -> materialized partition
  t=8: idle
  t=9: idle

sensor fired at ticks: [3, 7]


## Freshness: when is an asset too old?

A freshness policy sets a maximum age. If the newest materialization of an asset is older than that, the asset is **stale** and anything reading it is on thin ice. This is how a platform turns "the dashboard looks wrong" into an alert that fires before anyone opens the dashboard.

In [3]:
@dataclass
class FreshnessPolicy:
    max_age: int   # in clock ticks

    def is_stale(self, last_materialized, now):
        if last_materialized is None:
            return True
        return (now - last_materialized) > self.max_age


policy = FreshnessPolicy(max_age=4)
last = max(runs)            # asset last built at the sensor's final firing
print(f"asset last materialized at t={last}, policy allows age <= {policy.max_age}")
print()
for now in (last + 2, last + 4, last + 6):
    state = "STALE" if policy.is_stale(last, now) else "fresh"
    print(f"  at t={now} (age {now - last}): {state}")

asset last materialized at t=7, policy allows age <= 4

  at t=9 (age 2): fresh
  at t=11 (age 4): fresh
  at t=13 (age 6): STALE


Sensors and freshness are the reactive half of orchestration. A scheduler that only runs on a clock is blind to whether its inputs actually arrived. Next: point the orchestrator at the repo's own dbt project and let it discover the graph for itself.